# 01 — The frame, and the window chosen by rule

Everything real in ATHAR rests on this notebook. Olist gives 99,441 orders across
two years of Brazilian e-commerce; what comes out is a weekly revenue series, a
state-by-week panel, and a per-customer summary.

Two defects bound the usable history, and neither is a business fact. Collection
does not begin cleanly — the 2016 weeks are interrupted by a hole where no orders
were recorded at all — and it stops mid-week at the end, so the final weeks tail
off toward a single order. Choosing a window by eye would be indefensible, so it
is chosen by a rule stated before the data was inspected.

In [1]:
import warnings

import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [2]:
frame = read_metric("frame", METRICS)
print(frame["window"]["rule"])
print()
gaps = frame["verification"]["zero_order_weeks_in_window"]
print("window:", frame["window"]["start"], "to", frame["window"]["end"],
      f"({frame['window']['weeks']} weeks, {gaps} with no orders)")

Starts the week after the last week with zero orders, which removes the nine-week collection gap in late 2016. Ends at the last week carrying at least 70% of the median of the 8 weeks before it, which removes the mid-week truncation of the extract. The comparison is against recent weeks rather than the whole series because Olist grew throughout, so a half-collected final week can still sit above the global median.

window: 2017-01-02 to 2018-08-13 (85 weeks, 0 with no orders)


## What the rule keeps

98.5% of orders and 98.7% of revenue survive, and the window contains no gaps.

In [3]:
totals = frame["totals"]
coverage = frame["coverage"]
show(pd.DataFrame([
    {"quantity": "orders", "in window": totals["orders"], "all dates": coverage["orders_all_dates"],
     "share kept": round(coverage["share_of_orders_in_window"], 4)},
    {"quantity": "revenue (BRL)", "in window": round(totals["revenue_brl"], 2),
     "all dates": round(coverage["revenue_all_dates_brl"], 2),
     "share kept": round(coverage["share_of_revenue_in_window"], 4)},
]))
print("mean order value:", totals["mean_order_value_brl"], "BRL")
print("states:", totals["states"], " people:", totals["people"])

     quantity   in window   all dates  share kept
       orders    96731.00    98199.00      0.9851
revenue (BRL) 13312577.43 13494400.74      0.9865

mean order value: 137.62 BRL
states: 27  people: 93573


## Verification: an independent recomputation

`athar.frame` stages the join through CTEs; the check in `scripts/build_frame.py`
uses a flat join and filter, sharing no code path. The two must agree.

They agree on the order count exactly and on revenue to 1.0e-12 relative — not
bit-for-bit, and they cannot be. Float addition is not associative, the two plans
accumulate 96,731 terms in different orders, and DuckDB is already pinned to one
thread. So the test is agreement to the centavo, which is the resolution the
quantity actually has.

In [4]:
verification = frame["verification"]
for key, value in verification.items():
    print(f"{key:52s} {value}")

independent_order_count_matches                      True
independent_revenue_difference_brl                   1.37817e-05
independent_revenue_relative_difference              1.035e-12
independent_revenue_tolerance_brl                    0.01
zero_order_weeks_in_window                           0


## The geography, and the power problem it creates

One state carries 38% of revenue; thirteen of the twenty-seven carry under 1%
each. That concentration is what makes a geo holdout expensive, and it is Olist's,
not this project's invention. Notebook 05 measures the consequence.

In [5]:
geo = frame["geo_panel"]
print("states:", geo["states"], "| balanced panel:", geo["balanced"], "| rows:", geo["rows"])
print("largest state share of revenue:", round(geo["largest_state_revenue_share"], 4))
print("states below 1% of revenue:", geo["states_below_one_percent_of_revenue"])

states: 27 | balanced panel: True | rows: 2295
largest state share of revenue: 0.3808
states below 1% of revenue: 13


## The repeat rate, which decides notebook 06

A person is `customer_unique_id`. Olist issues a fresh `customer_id` per order, so
keying on that would report a repeat rate of exactly zero — a mistake that would
have made the CLV notebook meaningless rather than merely difficult.

In [6]:
repeat = frame["repeat_behaviour"]
show(pd.DataFrame([
    {"basis": "all dates", **repeat["all_dates"]},
    {"basis": "in window", **{k: v for k, v in repeat["in_window"].items()}},
]))
print(repeat["note"])

    basis  people  repeat_rate  with_repeat  with_two_or_more_repeats
all dates   94983     0.030395         2887                       NaN
in window   93573     0.030329         2838                     230.0

A person is customer_unique_id; Olist issues a fresh customer_id per order, so keying on that would report a repeat rate of exactly zero.


**Carried forward.** A gap-free 85-week weekly series with a real trend and real
seasonality, which notebook 02 uses as the non-media baseline; a balanced 27-state
panel for notebook 05; and a 3% repeat rate that notebook 06 has to take seriously.